# 🔬 RedPitayaSTCL: Setup & Compatibility Check

<p style="font-size:1.05em; color:#444;">
This notebook verifies that your <strong>PC environment</strong> and each
<strong>RedPitaya board</strong> meet all requirements before running the
Scanning Transfer Cavity Lock (STCL) system.
</p>

<blockquote style="border-left:4px solid #e67e22; padding:6px 12px;
  background:#fdf6ec; color:#7f4f00; border-radius:4px;">
  <strong>→ Run all cells top-to-bottom on a fresh kernel before any locking workflow. Remember to update ip-address in the code cell below.</strong>
</blockquote>

<br>

| Step | What it does |
|------|-------------|
| **①** | Configuration — set board IPs here |
| **②** | Import the checker module |
| **③** | PC environment checks |
| **④** | Board information (SSH) |
| **⑤** | Board compatibility checks |
| **⑥** | Summary report |


---
## 1) Configuration

Edit the cell below **before running anything else**. Add one entry per physical RedPitaya board.

| Key | Example | Description |
|-----|---------|-------------|
| `ip` | `"192.168.0.101"` | Board IP address on your network |
| `mode` | `"scan"` | One of: `scan` · `lock` · `monitor` |

<br>

| Mode | Role | Outputs used |
|------|------|-------------|
| `scan` | Generates cavity scan ramp + trigger square wave | OUT2 → ramp, OUT1 → trigger |
| `lock` | Applies PID feedback to laser current mod inputs | OUT1 → Slave1, OUT2 → Slave2 |
| `monitor` | Passive cavity signal monitor (no outputs) | — |


In [9]:
# ── Edit here ────────────────────────────────────────────────────────────────
BOARDS = {
    "Cav"  : {"ip": "192.168.0.99", "mode": "scan"},
    # "Lock1": {"ip": "192.168.0.102", "mode": "lock"},
    # "Mon"  : {"ip": "192.168.0.100", "mode": "monitor"},
}

SSH_USER       = "root"
SSH_PASS       = "root"
SSH_PORT       = 22

STCL_CMD_PORT  = 5000   # RP_Server command listener
STCL_LOOP_PORT = 5065   # reaction_loop port (opened during lock / scan)

---
## 2) Import the Checker Module

All check logic lives in `setup.py` (same directory as this notebook).
The cell below locates the repo root, adds it to `sys.path`, and imports the module.

> **If you see `ModuleNotFoundError: No module named 'setup'`** — make sure
> `setup.py` is in the same folder as this notebook, or add the path manually:
> `sys.path.insert(0, '/path/to/RedPitayaSTCL')`


In [10]:
import sys, pathlib

# Automatically find the repo root (folder containing setup.py)
_nb_dir = pathlib.Path().resolve()
for _candidate in [_nb_dir] + list(_nb_dir.parents)[:3]:
    if (_candidate / "setup.py").exists() and (_candidate / "lockclient.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import setup                    # RedPitayaSTCL setup.py
import importlib; importlib.reload(setup)   # pick up edits without restarting kernel
print("setup.py loaded from:", pathlib.Path(setup.__file__).resolve())

setup.py loaded from: C:\Users\Qulabs\Projects\RP-STCL\setup.py


---
## 3) PC Environment Checks

Verifies the Python version, all required packages, the `Qt5Agg` matplotlib
backend, and that the repository is correctly importable. All checks must pass
before connecting to any board.

| Package | Used for |
|---------|----------|
| `paramiko` | SSH/SFTP — upload scripts and start server on each RP |
| `numpy` | Signal processing and array operations |
| `scipy` | Golden-ratio figure sizing; Savitzky-Golay filter math |
| `matplotlib` | Qt5Agg backend for live cavity and error monitor windows |


<br>

> **If Qt5Agg is missing:** `pip install PyQt5`


In [11]:
pc_ok = setup.check_pc()

────────────────────────────────────────────────────────────────────
  PC ENVIRONMENT CHECKS
────────────────────────────────────────────────────────────────────
  ✓ Python version  →  3.14.3 (>= 3.7 required for f-strings and other PC-side syntax)
  ✓ Package: paramiko      →  v4.0.0  —  SSH/SFTP: uploads scripts and starts server on each RP
  ✓ Package: numpy         →  v2.4.3  —  Signal processing and array operations
  ✓ Package: scipy         →  v1.17.1  —  Golden-ratio figure sizing; Savitzky-Golay filter math
  ✓ Package: matplotlib    →  v3.10.8  —  Qt5Agg backend for live cavity and error monitor windows
  ✓ matplotlib backend  →  Qt5Agg available — live monitor windows will work
  ✓ Repo on sys.path  →  C:\Users\Qulabs\Projects\RP-STCL
  ✓ RP_side importable  →  peak_finders module imported successfully
  ✓ RP_side/RP_Lock.py            →  Main locking loop (uploaded to board)
  ✓ RP_side/RunLock.py            →  Entry point executed on board via SSH
  ✓ RP_side/libserver.py 

---
## 4) Board Information

<p>Connects to each board via SSH and collects full hardware and software details
<em>before</em> any pass/fail judgement is applied.
Read the raw output here to understand exactly what is installed on each board.</p>

<blockquote style="border-left:4px solid #2980b9; padding:6px 12px;
  background:#eaf4fb; color:#1a5276; border-radius:4px;">
  <strong>What is collected:</strong>
  OS version · RP ecosystem version · hostname · uptime · CPU · RAM ·
  Python version &amp; sys.path · numpy version · rp SWIG module path &amp; import test ·
  STCL files on board · port 5000/5065 status ·
  stale RunLock processes · active system services
</blockquote>

<br>

> **SSH note:** The board runs Ubuntu 22.04 (RP OS 2.x). Standard paramiko
> key-exchange works without any `disabled_algorithms` workaround.


In [12]:
import setup   # ensure latest version

board_infos = {}
for name, cfg in BOARDS.items():
    info = setup.collect_board_info(
        name, cfg["ip"], cfg["mode"],
        ssh_user=SSH_USER, ssh_pass=SSH_PASS, ssh_port=SSH_PORT,
        stcl_cmd_port=STCL_CMD_PORT, stcl_loop_port=STCL_LOOP_PORT,
    )
    board_infos[name] = info

────────────────────────────────────────────────────────────────────
  Board: Cav  (192.168.0.99)  mode=scan
────────────────────────────────────────────────────────────────────
  ✓ Ping 192.168.0.99  — reachable
  ✓ SSH login  root@192.168.0.99
  i OS                : Ubuntu 22.04.5 LTS
  i RP ecosystem      : 2.07-ffe70f24f
  i RP .version file  : 2.07
  i Hostname          : rp-f0c97c
  i Uptime            : up 1 hour, 26 minutes
  i CPU               : ARMv7 Processor rev 0 (v7l)
  i Memory            : 461 MB total, 90 MB used
  i Python            : Python 3.10.12  (/usr/bin/python3)
  i /opt/redpitaya/lib/python on board sys.path: NO
  i numpy             : 2.2.5
  i rp module path    : not found
  i rp module import  : ok
  i /opt/redpitaya/bin : present
  i /opt/redpitaya/fpga: present
  ✓ Board STCL file  : /root/RP_Lock.py
  ✓ Board STCL file  : /root/RunLock.py
  ✓ Board STCL file  : /root/libserver.py
  ✓ Board STCL file  : /root/peak_finders.py
  ✓ Port 5000 (cmd)  : free

---
## 5) Board Compatibility Checks

Evaluates the board information against the known requirements of the STCL
codebase. Each result is marked:

| Symbol | Meaning |
|--------|---------|
| `✓` | **Pass**: requirement met |
| `⚠` | **Warning**: may work but needs attention |
| `✗` | **Fail**: must be resolved before running STCL |

<br>

<details>
<summary><strong>Common failures and fixes (click to expand)</strong></summary>

<br>

| Issue | Fix |
|-------|-----|
| RP OS version is 1.04 | Flash SD card with OS 2.x from `downloads.redpitaya.com` |
| `rp` module not found / import fails | Verify `/opt/redpitaya/lib/python/rp.py` exists; check FPGA overlay is loaded |
| numpy ≥ 2.0 on board | Use patched `peak_finders.py` from this repo (`np.mat` → `np.array`) |
| Port 5000 in use | `pkill -f RunLock.py` on the board |
| SCPI service active | `systemctl stop redpitaya_scpi` on the board |
| STCL files not uploaded | Run `LockClient(RPs)` — `upload_current()` transfers them to `/root/` via SFTP |

</details>


In [13]:
board_ok_all = True
for name, info in board_infos.items():
    ok = setup.check_board(info,
                           stcl_cmd_port=STCL_CMD_PORT,
                           stcl_loop_port=STCL_LOOP_PORT)
    board_ok_all = board_ok_all and ok

────────────────────────────────────────────────────────────────────
  Compatibility checks: Cav  (192.168.0.99)
────────────────────────────────────────────────────────────────────
  ✓ OS version  →  Ubuntu 22.04 LTS — matches the tested STCL OS 2.x configuration
  ✓ RP ecosystem version  →  2.07-ffe70f24f — OS 2.x confirmed. rp SWIG module supported.
  ✓ Board Python version  →  Python 3.10.12 — OS 2.x ships Python 3.10; RP-side code compatible
  ⚠ rp lib on board sys.path  →  /opt/redpitaya/lib/python NOT in python3 sys.path. RP_Lock.py uses PYTHONPATH=/opt/redpitaya/lib/python/:$PYTHONPATH in the SSH launch command — this is handled automatically by communication.py start_host_server(). No manual action needed unless you launch RunLock.py by hand.
  ⚠ Board numpy version  →  v2.2.5 — numpy >= 2.0 removed np.mat (used in peak_finders.py). Confirm you are using the patched peak_finders.py from this repo (np.mat replaced with np.array).
  ✗ rp module  →  /opt/redpitaya/lib/python/rp.p

---
## 6) Summary Report

Aggregates all results across PC and boards into a single final verdict.

- All `✗ FAIL` items must be resolved before attempting any locking workflow.
- `⚠ WARNING` items should be reviewed — they may cause subtle issues at runtime.
- A green `✓` at the bottom means the system is ready.

> After resolving any issues, **restart the kernel and re-run all cells**
> to confirm a clean state.


In [14]:
system_ready = setup.print_summary()


════════════════════════════════════════════════════════════════════
  SUMMARY
════════════════════════════════════════════════════════════════════

  ✓ PC environment
    13 passed  |  0 warnings  |  0 failed

  ✗ Board: Cav  (192.168.0.99)
    12 passed  |  3 warnings  |  1 failed
    Failed:
        ✗ rp module  →  /opt/redpitaya/lib/python/rp.py not found. This file is part of the RedPitaya OS 2.x installation. Verify the board is running OS 2.x and /opt/redpitaya/lib/python/ exists.
    Warnings:
        ⚠ rp lib on board sys.path  →  /opt/redpitaya/lib/python NOT in python3 sys.path. RP_Lock.py uses PYTHONPATH=/opt/redpitaya/lib/python/:$PYTHONPATH in the SSH launch command — this is handled automatically by communication.py start_host_server(). No manual action needed unless you launch RunLock.py by hand.
        ⚠ Board numpy version  →  v2.2.5 — numpy >= 2.0 removed np.mat (used in peak_finders.py). Confirm you are using the patched peak_finders.py from this repo (np.mat repl